## CAPSTONE PROJECT
### 01 | AI Data Analyst

* Analyze student performance data using Pandas, SQL, and Groq AI. Answer natural language questions about the data automatically.
* **Tools:** Pandas + Groq API


In [1]:
!pip install groq pandas -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.6 MB/s eta 0:00:00


In [2]:
import pandas as pd

df = pd.read_csv("student_performance (3).csv")

df.head()

,student_id,name,age,gender,branch,attendance_pct,assignment_score,midterm_score,final_score,gpa,passed
0,1,Aarav Sharma,20,Male,CSE,85,78,72,76,7.6,Yes
1,2,Priya Patel,21,Female,ECE,92,88,85,89,8.9,Yes
2,3,Rohit Kumar,20,Male,MECH,67,55,60,58,5.8,Yes
3,4,Sneha Iyer,22,Female,CSE,95,92,90,94,9.4,Yes
4,5,Vikram Singh,21,Male,CIVIL,72,62,65,63,6.3,Yes


In [3]:
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

(30, 11)

Columns:
Index(['student_id', 'name', 'age', 'gender', 'branch', 'attendance_pct',
       'assignment_score', 'midterm_score', 'final_score', 'gpa', 'passed'],
      dtype='object')

Data Types:
student_id            int64
name                 object
age                   int64
gender               object
branch               object
attendance_pct        int64
assignment_score      int64
midterm_score         int64
final_score           int64
gpa                 float64
passed               object
dtype: object


In [4]:
print("Average GPA:", df["gpa"].mean())

print("Average Attendance:", df["attendance_pct"].mean())

print("Highest GPA:", df["gpa"].max())

print("Lowest GPA:", df["gpa"].min())

Average GPA: 7.293333333333334
Average Attendance: 78.63333333333334
Highest GPA: 9.5
Lowest GPA: 4.0


In [5]:
branch_analysis = df.groupby("branch").agg(
    avg_gpa=("gpa","mean"),
    avg_attendance=("attendance_pct","mean"),
    students=("student_id","count")
)

branch_analysis.reset_index(inplace=True)

branch_analysis

,branch,avg_gpa,avg_attendance,students
0,CIVIL,6.750000,75.000000,4
1,CSE,7.420000,80.000000,10
2,ECE,6.383333,69.166667,6
3,IT,8.640000,89.400000,5
4,MECH,7.220000,79.400000,5


In [6]:
top_students = df.sort_values(
    by="gpa",
    ascending=False
).head(5)

top_students

,student_id,name,age,gender,branch,attendance_pct,assignment_score,midterm_score,final_score,gpa,passed
9,10,Meera Krishnan,22,Female,IT,96,94,92,95,9.5,Yes
3,4,Sneha Iyer,22,Female,CSE,95,92,90,94,9.4,Yes
21,22,Lakshmi Chandran,22,Female,CSE,94,91,89,92,9.2,Yes
17,18,Swathi Menon,20,Female,IT,93,90,88,91,9.1,Yes
1,2,Priya Patel,21,Female,ECE,92,88,85,89,8.9,Yes


In [12]:
from groq import Groq

client = Groq(
    api_key="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"
)

In [13]:
data_summary = ""

data_summary += f"""
Total Students: {len(df)}
Average GPA: {df['gpa'].mean():.2f}
Average Attendance: {df['attendance_pct'].mean():.2f}
"""

data_summary += "\n\nBranch Analysis:\n"

for _, row in branch_analysis.iterrows():

    data_summary += (
        f"- {row['branch']} : "
        f"{row['students']} students, "
        f"Avg GPA {row['avg_gpa']:.2f}, "
        f"Avg Attendance {row['avg_attendance']:.2f}%\n"
    )

data_summary += "\nTop Students:\n"

for _, row in top_students.iterrows():

    data_summary += (
        f"- {row['name']} ({row['branch']}) "
        f"GPA {row['gpa']}\n"
    )

print(data_summary)


Total Students: 30
Average GPA: 7.29
Average Attendance: 78.63


Branch Analysis:
- CIVIL : 4 students, Avg GPA 6.75, Avg Attendance 75.00%
- CSE : 10 students, Avg GPA 7.42, Avg Attendance 80.00%
- ECE : 6 students, Avg GPA 6.38, Avg Attendance 69.17%
- IT : 5 students, Avg GPA 8.64, Avg Attendance 89.40%
- MECH : 5 students, Avg GPA 7.22, Avg Attendance 79.40%

Top Students:
- Meera Krishnan (IT) GPA 9.5
- Sneha Iyer (CSE) GPA 9.4
- Lakshmi Chandran (CSE) GPA 9.2
- Swathi Menon (IT) GPA 9.1
- Priya Patel (ECE) GPA 8.9



In [14]:
responsible_system_prompt = """
You are a helpful data analysis assistant.

You only answer questions based on the student dataset summary provided.

If the answer is not available from the data,
say:

'I do not have enough information to answer that accurately.'

Do not invent facts or statistics.
"""

In [15]:
def ask_ai(question):

    prompt = f"""
Dataset Summary:

{data_summary}

Question:
{question}
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role":"system",
                "content":responsible_system_prompt
            },
            {
                "role":"user",
                "content":prompt
            }
        ],
        temperature=0.3,
        max_tokens=300
    )

    return response.choices[0].message.content

In [16]:
print(
    ask_ai(
        "Which branch has the highest average GPA?"
    )
)

The IT branch has the highest average GPA, with an average GPA of 8.64.


In [17]:
print(
    ask_ai(
        "Who are the top students?"
    )
)

The top students are:

1. Meera Krishnan (IT) with a GPA of 9.5
2. Sneha Iyer (CSE) with a GPA of 9.4
3. Lakshmi Chandran (CSE) with a GPA of 9.2
4. Swathi Menon (IT) with a GPA of 9.1
5. Priya Patel (ECE) with a GPA of 8.9


In [18]:
print(
    ask_ai(
        "Which branch has the best attendance?"
    )
)

The IT branch has the best attendance, with an average attendance of 89.40%.


In [19]:
print(
    ask_ai(
        "Give me insights from the dataset."
    )
)

Based on the provided dataset summary, here are some insights:

1. **Overall Performance**: The average GPA of all students is 7.29, and the average attendance is 78.63%. This suggests that students are performing reasonably well academically, but there may be room for improvement in terms of attendance.

2. **Branch-wise Performance**: The IT branch has the highest average GPA (8.64) and attendance (89.40%), indicating that IT students are performing exceptionally well. In contrast, the ECE branch has the lowest average GPA (6.38) and attendance (69.17%), suggesting that ECE students may need additional support.

3. **Top-performing Students**: The top 5 students are from the IT, CSE, and ECE branches, with Meera Krishnan (IT) having the highest GPA (9.5). This suggests that these branches have some outstanding students who are excelling academically.

4. **Branch-wise Student Distribution**: The CSE branch has the most students (10), followed by ECE (6), and then the other branches h

In [20]:
while True:

    question = input("\nAsk a question: ")

    if question.lower() == "exit":
        break

    print("\nAI Analyst:\n")

    print(ask_ai(question))


Ask a question: Give me 2 insights from the dataset

AI Analyst:

Based on the dataset summary, here are two insights:

1. The IT branch has the highest average GPA (8.64) and average attendance (89.40%), indicating that students in this branch tend to perform well academically and have good attendance records.
2. The top 5 students are dominated by students from the CSE and IT branches, with 2 students from CSE (Sneha Iyer and Lakshmi Chandran) and 2 students from IT (Meera Krishnan and Swathi Menon), suggesting that these branches may have a higher proportion of high-achieving students.

Ask a question: what is the average gpa

AI Analyst:

The average GPA is 7.29.

Ask a question: exit
